# 01 — Understand the retail source

Start here in a fresh kernel. One row is an invoice line, not a complete order. Read PROBLEM_STATEMENT.md and notebooks/README.md. This notebook only reads the source.

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(folder for folder in (Path.cwd(), *Path.cwd().parents)
            if (folder / "PROBLEM_STATEMENT.md").is_file())
RAW = ROOT / "data/raw/online_retail_II.csv"
PROCESSED = ROOT / "data/processed"
QUALITY = ROOT / "data/quality"
TABLES = ROOT / "power_bi/tables"
SOURCE_COLUMNS = ["Invoice", "StockCode", "Description", "Quantity", "InvoiceDate", "Price", "CustomerID", "Country"]
FACT_COLUMNS = ["SourceRow", *SOURCE_COLUMNS, "InvoiceDay", "RevenuePence", "TotalAmount", "Year", "Month", "MonthName", "DayOfWeek", "WeekdayNumber", "Hour"]
SOURCE = RAW  # For the original workbook, use ROOT / "data/raw/online_retail_II.xlsx".

## 1. Read the complete source

The XLSX loader combines both annual sheets. A CSV containing only one sheet is incomplete. Customer identifiers and invoice numbers are treated as text.

In [ ]:
def load_raw(path: Path = RAW) -> pd.DataFrame:
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(
            f"Source not found: {path}. See README.md, Reproduce the analysis."
        )
    if path.suffix.lower() == ".xlsx":
        # UCI's workbook has two annual sheets; reading only the first loses a year.
        frame = pd.concat(
            pd.read_excel(path, sheet_name=None, dtype="string").values(),
            ignore_index=True,
        )
    else:
        frame = pd.read_csv(path, dtype="string")
    frame.columns = frame.columns.str.strip().str.replace(" ", "", regex=False)
    frame = frame.rename(columns={"InvoiceNo": "Invoice", "UnitPrice": "Price"})
    missing = set(SOURCE_COLUMNS) - set(frame.columns)
    if missing or frame.columns.duplicated().any():
        raise ValueError(
            f"Invalid source columns; missing={sorted(missing)} or duplicate headers."
        )
    return frame[SOURCE_COLUMNS].copy()

In [ ]:
raw = load_raw(SOURCE)
with SOURCE.open("rb") as stream:
    source_hash = hashlib.file_digest(stream, "sha256").hexdigest()
manifest = json.loads((ROOT / "data/raw/manifest.json").read_text(encoding="utf-8"))
print(f"Source rows: {len(raw):,}")
print("SHA-256:", source_hash)
print("Matches the recorded local CSV:", source_hash == manifest["sha256"])
display(raw.head())

## 2. Inspect quality and coverage

Missing customers change the population. Cancellations are outside positive sales. Exact repeated source rows may be genuine repeat lines because there is no unique line identifier.

In [ ]:
display(raw.isna().sum().rename("Missing rows").to_frame())
dates = pd.to_datetime(raw.InvoiceDate, errors="coerce", format="mixed")
print("Coverage:", dates.min(), "to", dates.max())
print("Cancelled lines:", raw.Invoice.str.upper().str.startswith("C").sum())
print("Exact repeats before trimming:", raw.duplicated().sum())
display(raw.groupby("Country").size().sort_values(ascending=False).head(10).rename("Source lines").to_frame())

## Explain before continuing

Explain the difference between lines, orders and customers. Why do missing-customer exclusions change the analysis? Why is December 2011 incomplete? Why can this project report positive sales but not profit? Continue to notebook 02 after inspecting the source hash and coverage.